# WM-811K robust-flat bias 적용 후 견고성 재검증

Validation 예측에서 세 seed 공통으로 선택한 none logit bias `1.155`를 고정합니다. 4px 이동과 1% 활성 die 누락 조건을 다시 추론하며 **test 추론은 수행하지 않습니다**. 재학습은 없습니다.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import json, shutil, subprocess, sys
import pandas as pd
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'T4 GPU 런타임을 선택한 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))
DATA_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_processed')
EXPERIMENT_ROOT = Path('/content/drive/MyDrive/AI_project/WM811K_robust_hierarchical/validation_multiseed')
OUTPUT_DIR = Path('/content/drive/MyDrive/AI_project/WM811K_robust_hierarchical/stress_replay_bias_1_155')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for path in (DATA_DIR/'wafer_maps_64.npy', DATA_DIR/'labels.npy', EXPERIMENT_ROOT/'runs'):
    assert path.exists(), f'필수 파일이 없습니다: {path}'


## 로컬 파일 4개를 차례로 선택

1. `train_wm811k_cnn.py`
2. `train_wm811k_robust_hierarchical_candidates.py`
3. `replay_wm811k_robust_flat_stress.py`
4. `split_assignments.csv`


In [ ]:
def upload_as(target_name):
    print(f'{target_name} 파일을 선택하세요.')
    uploaded = files.upload()
    assert len(uploaded) == 1, '이번 선택창에서는 파일 하나만 선택하세요.'
    _, content = next(iter(uploaded.items()))
    target = Path('/content') / target_name
    target.write_bytes(content)
    return target

BASE_SCRIPT = upload_as('train_wm811k_cnn.py')
ROBUST_SCRIPT = upload_as('train_wm811k_robust_hierarchical_candidates.py')
REPLAY_SCRIPT = upload_as('replay_wm811k_robust_flat_stress.py')
ASSIGNMENTS = upload_as('split_assignments.csv')


In [ ]:
command = [
    sys.executable, str(REPLAY_SCRIPT),
    '--maps', str(DATA_DIR/'wafer_maps_64.npy'),
    '--labels', str(DATA_DIR/'labels.npy'),
    '--assignments', str(ASSIGNMENTS),
    '--experiment-root', str(EXPERIMENT_ROOT),
    '--output-dir', str(OUTPUT_DIR),
    '--none-logit-bias', '1.155',
    '--batch-size', '512', '--num-workers', '2',
]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
return_code = process.wait()
assert return_code == 0, f'견고성 재검증 실패: exit code {return_code}'


In [ ]:
summary = json.loads((OUTPUT_DIR/'stress_replay_summary.json').read_text(encoding='utf-8'))
display(pd.read_csv(OUTPUT_DIR/'stress_replay_metrics.csv'))
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert summary['test_evaluated'] is False
print('배포 모델 변경:', summary['deployment_model_changed'])


In [ ]:
EXPORT_DIR = Path('/content/wm811k_robust_flat_stress_replay_results')
if EXPORT_DIR.exists(): shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir()
for name in ('stress_replay_summary.json', 'stress_replay_metrics.csv'):
    shutil.copy2(OUTPUT_DIR/name, EXPORT_DIR/name)
zip_path = Path(shutil.make_archive(str(EXPORT_DIR), 'zip', root_dir=EXPORT_DIR))
print('다운로드:', zip_path)
files.download(str(zip_path))
